In [ ]:
import yaml
import numpy as np
import polars as pl
import polars.selectors as cs
from tqdm import tqdm
import statsmodels.api as sm
from scipy import stats

from plotnine import *
import matplotlib.pyplot as plt
plt.rcParams['svg.fonttype'] = 'none'

In [5]:
# Get gene trait associations
RAP_DIR = 'project-Gyp4fvjJg0yFZ374KvP9bGFJ:/processed_data/REGENIE_results'

# ASSOC_FILE = 'loftee_mac20_associations_bh_corrected.parquet'
# ASSOC_FILE = 'regenie_127phenotypes_lofteeHC_mac20_EUR.parquet'
ASSOC_FILE = 'regenie_127phenotypes_lofteeHC_mac20_EUR_miss20per.parquet'

LOCAL_DIR = '/home/dnanexus/data_dir/'

!dx download {RAP_DIR}/{ASSOC_FILE} -o {LOCAL_DIR}

gene_trait_df = (
    pl.read_parquet(f'{LOCAL_DIR}/{ASSOC_FILE}')
    .filter(pl.col('pval_fdr')<=0.05)
    .select(['region', 'phenotype', 'pval_fdr'])
)

# CORR_FILE = 'regenie_127phenotypes_mac20_lofteeHC_EUR_correlations.parquet'
CORR_FILE = "regenie_127phenotypes_lofteeHC_mac20_EUR_miss20per_correlations.parquet"
!dx download {RAP_DIR}/{CORR_FILE} -o {LOCAL_DIR}

loftee_corrs = (
    pl.read_parquet(f'{LOCAL_DIR}/{CORR_FILE}')
    .with_columns(
        loftee_corr = pl.col('correlation'),
        loftee_corr_abs = pl.col('correlation').abs(),
        loftee_corr_dir = pl.col('correlation')/pl.col('correlation').abs(),
    )
    .select(['region', 'phenotype', 'loftee_corr', 'loftee_corr_abs', 'loftee_corr_dir']) 
)

gene_trait_df = (
    gene_trait_df
    .join(loftee_corrs, on=['region', 'phenotype'], how='inner')
    .drop_nans()
    .sort('loftee_corr_abs', descending=True)
    .unique(subset=["region"], keep="first", maintain_order=True)
)
gene_trait_df

Error: path "/home/dnanexus/data_dir/regenie_127phenotypes_lofteeHC_mac20_EUR_
miss20per.parquet" already exists but -f/--overwrite was not set
Error: path "/home/dnanexus/data_dir/regenie_127phenotypes_lofteeHC_mac20_EUR_
miss20per_correlations.parquet" already exists but -f/--overwrite was not set


region,phenotype,pval_fdr,loftee_corr,loftee_corr_abs,loftee_corr_dir
str,str,f64,f64,f64,f64
"""ENSG00000084674""","""ldl_direct_int""",2.2543e-301,-0.12678,0.12678,-1.0
"""ENSG00000167701""","""alanine_aminotransferase_int""",1.2234e-144,-0.106424,0.106424,-1.0
"""ENSG00000105610""","""mean_corpuscular_haemoglobin_i…",9.8993e-24,-0.100921,0.100921,-1.0
"""ENSG00000141505""","""alkaline_phosphatase_int""",8.4146e-35,0.089171,0.089171,1.0
"""ENSG00000101162""","""platelet_distribution_width_in…",1.5335e-83,0.08869,0.08869,1.0
…,…,…,…,…,…
"""ENSG00000115977""","""sitting_height_int""",0.021257,-0.001496,0.001496,-1.0
"""ENSG00000119574""","""standing_height_int""",0.025535,-0.001317,0.001317,-1.0
"""ENSG00000115325""","""platelet_distribution_width_in…",0.01277,-0.001204,0.001204,-1.0


In [7]:
# Configuration and paths
mac = 20

# Maximum number of top-ranked variants per annotation (None = no limit)
max_num_variants = None

# Initialize new annotation variable as None
new_anno_local = None

# Load annotation configuration
config_path = "/home/dnanexus/ukbgym/config_odds.yaml"

with open(config_path) as f:
    config = yaml.safe_load(f)

eur_samples_path = '/home/dnanexus/data_dir/unrelated_cauc_samples_3rd_degree.csv'

records = [
    {
        "category": category,
        "annotation": anno,
        "color": props["color"],
        "label": props["label"],
        "annotation_dir": props.get("direction", 1),
    }
    for category, annos in config["rare_variant_annotations"].items()
    for anno, props in annos.items()
]

# Create the DataFrame directly from the list of records
anno_config_df = pl.DataFrame(records).with_columns(
    pl.col("annotation_dir").cast(pl.Int8)
)

all_annotation_list = anno_config_df.select(pl.col("annotation")).to_series().to_list()

anno_config_df

category,annotation,color,label,annotation_dir
str,str,str,str,i8
"""missense""","""am_pathogenicity""","""#FFD700""","""AlphaMissense""",1
"""missense""","""esmscoremissense""","""#FFA500""","""ESM1v""",-1
"""missense""","""esm1b_llr""","""#FF6600""","""ESM1b""",-1
"""missense""","""cpt1_llr""","""#E63000""","""CPT-1""",1
"""missense""","""revel_score""","""darkred""","""REVEL""",1
…,…,…,…,…
"""regulatory""","""fz_all_mean""","""#1f77b4""","""Flashzoi mean across RNA (all)""",1
"""regulatory""","""fz_blood_min""","""crimson""","""Flashzoi min across RNA (blood…",-1
"""regulatory""","""fz_all_min""","""#FFA500""","""Flashzoi min across RNA (all)""",-1


In [ ]:
RAP_ANNO_DIR = "project-Gyp4fvjJg0yFZ374KvP9bGFJ:/processed_data/wgs/qced_maf1e-3_loftee_olink_genes/qced_maf1e-3_loftee_olink_genes_EURunrelated"
LOCAL_ANNO_DIR = "/home/dnanexus/data_dir"

# ANNO_FILE = "annotations_fillna_ukbgym_with_mane.parquet"
ANNO_FILE = "annotations_with_all.parquet"

!dx download {RAP_ANNO_DIR}/{ANNO_FILE} -o {LOCAL_ANNO_DIR}/{ANNO_FILE}
anno = pl.scan_parquet(f"{LOCAL_ANNO_DIR}/{ANNO_FILE}")

anno = (
    anno
    .filter(
        # Filter to gene regions of interest
        (pl.col('region').is_in(gene_trait_df['region'].unique())),
        
        # Choose Gene Body
        # ((pl.col('consequence_upstream_gene_variant') == False) & (pl.col('consequence_downstream_gene_variant') == False)),

        # Choose CDS
        # (pl.col('vep_cds_relaxed')==True),

        # Choose VEP consequence
        (pl.col('consequence_missense_variant') == True),

        # Choose protein domain
        # (pl.col('ted_domain') == True),
        # (pl.col('ted_domain') == False),
        # (pl.col('mobi_full_disorder_priority') == True),
        # (pl.col('mobi_curated_disorder_priority') == True),
        # (pl.col('mobi_full_lip_priority') == True),

        # Choose non CDS only
        # UPDATE WITH NEW VEP MANE ANNOS
        # ((pl.col('vep_cds_relaxed')==False) & (pl.col('mane_cds')==False) & (pl.col('non_mane_cds')==False)),

        # Choose Introns with/without alternate CDS
        # (pl.col('consequence_intron_variant') == True),

        # Core promoter
        # (pl.col('promoterai_is_na') == False),

        # Choose regulatory region
        # (pl.col('pangolin_score') < 0.2), # Remove splice variants from the regulatory analysis
        # (pl.col('encode_any_tf') == True),
        # (pl.col('encode_enhancer') == True),
        # (pl.col('encode_promoter') == True),

        # Only SNPs
        (pl.col('ref').str.len_chars()==1) & (pl.col('alt').str.len_chars()==1)
    )
)


selected_categories = ['missense'] # missense
# selected_categories = ['splicing'] # splicing

selected_annos = anno_config_df.filter(
    pl.col('category').is_in(selected_categories)
)['annotation'].to_list()

existing_annos = [c for c in all_annotation_list if c in anno.collect_schema().names()]
selected_annos = list(set(selected_annos).intersection(set(existing_annos)))
fillna_cols = [c+'_is_na' for c in selected_annos]

anno = (
    anno
    .select(
        set(['id', 'region']).union(set(selected_annos))
    )
    .collect(engine='streaming')
    # .drop_nulls()
)

anno

Error: path "/home/dnanexus/data_dir/annotations_with_all.parquet" already
exists but -f/--overwrite was not set


/tmp/ipykernel_545252/2166903644.py:68: DeprecationWarning: `is_in` with a collection of the same datatype is ambiguous and deprecated.
Please use `implode` to return to previous behavior.

See https://github.com/pola-rs/polars/issues/22149 for more information.
/tmp/ipykernel_545252/2166903644.py:77: DeprecationWarning: `is_in` with a collection of the same datatype is ambiguous and deprecated.
Please use `implode` to return to previous behavior.

See https://github.com/pola-rs/polars/issues/22149 for more information.


polyphen,revel_score,am_pathogenicity,esm1b_llr,cpt1_llr,id,esmscoremissense,region
f32,f64,f32,f64,f64,str,f32,str
0.881,0.152,0.7173,-7.053,0.225474,"""chr5:96432941:T:G""",-5.254,"""ENSG00000175426"""
0.112,0.104,0.2781,-6.579,0.27106,"""chr17:28700367:C:T""",-6.218,"""ENSG00000109111"""
0.297,0.119,0.0782,-5.308,0.212085,"""chr16:2092095:G:C""",-2.585,"""ENSG00000008710"""
0.583,0.608,0.0,null,null,"""chr6:165416273:A:G""",-6.679,"""ENSG00000112541"""
0.052,0.211,0.383,-8.595,0.124594,"""chr10:29463575:T:C""",-3.547,"""ENSG00000197321"""
…,…,…,…,…,…,…,…
0.0,0.124,0.0891,-5.92,0.095963,"""chr17:47596446:A:G""",-3.925,"""ENSG00000141279"""
0.04,0.102,0.9695,-2.447,0.093516,"""chr17:42838105:T:C""",-2.232,"""ENSG00000131467"""
0.04,0.102,0.9695,null,null,"""chr17:42838105:T:C""",-2.232,"""ENSG00000131467"""


In [9]:
anno_fillna = pl.scan_parquet(f"{LOCAL_ANNO_DIR}/{ANNO_FILE}")
fillna_cols = set([c+'_is_na' for c in selected_annos]).intersection(set(anno_fillna.collect_schema().names()))

anno_fillna_melted = (
    anno_fillna
    .filter(pl.col('region').is_in(gene_trait_df['region'].unique()))
    .select(['id', 'region'] + list(fillna_cols))
    .join(
        anno.select(['id', 'region']).lazy(), 
        on=['id', 'region'], 
        how='semi'
    )
    .unpivot(
        index=["id", "region"],
        on=list(fillna_cols),
        variable_name="annotation",
        value_name="annotation_is_na"
    )
    .filter(
        pl.col('annotation_is_na') == 1
    )
    .with_columns(
        annotation = pl.col('annotation').str.replace('_is_na$', '')
    )
)

anno_fillna_melted.head().collect()

/tmp/ipykernel_545252/2928459309.py:27: DeprecationWarning: `is_in` with a collection of the same datatype is ambiguous and deprecated.
Please use `implode` to return to previous behavior.

See https://github.com/pola-rs/polars/issues/22149 for more information.


id,region,annotation,annotation_is_na
str,str,str,i8
"""chr2:71276643:C:T""","""ENSG00000075292""","""esmscoremissense""",1
"""chr10:100735715:G:C""","""ENSG00000075891""","""esmscoremissense""",1
"""chr10:100735733:G:T""","""ENSG00000075891""","""esmscoremissense""",1
"""chr10:100735714:A:C""","""ENSG00000075891""","""esmscoremissense""",1
"""chr2:71276654:G:A""","""ENSG00000075292""","""esmscoremissense""",1


In [10]:
melted_anno = (
    anno.lazy()

    .unpivot(
        index=["id", "region"],
        on=selected_annos,
        variable_name="annotation",
        value_name="annotation_score"
    )
    .with_columns(
        pl.col("annotation_score").cast(pl.Float32),
        pl.col("region").cast(pl.Utf8),
    )

    # Merge with annotation configuration to get direction and filter
    .join(
        anno_config_df.select(["annotation", "category", "annotation_dir"]).lazy(),
        on="annotation",
        how="left"
    )
    .filter(pl.col('category').is_in(selected_categories))
    .unique()

    # Remove null annotation scores
    .drop_nulls('annotation_score')

    # Remove variants for each annotation which are not scored
    .join(
        anno_fillna_melted,
        on=['id', 'region', 'annotation'],
        how='anti'
    )

    # Correct scores by annotation direction
    .with_columns(
        annotation_score_dircor = pl.col('annotation_score') * pl.col("annotation_dir").cast(pl.Float32)
    )
    
    # Implement filtering based on the defined percentile
    .with_columns(
        annotation_score_dircor_rank_desc = pl.col('annotation_score_dircor').rank(method="max", descending=True).over(["annotation"]).cast(pl.Float32)
    )
    .drop(['annotation_score_dircor'])

    .collect(engine='streaming')
)

melted_anno

/tmp/ipykernel_545252/1598827054.py:45: DeprecationWarning: `is_in` with a collection of the same datatype is ambiguous and deprecated.
Please use `implode` to return to previous behavior.

See https://github.com/pola-rs/polars/issues/22149 for more information.


id,region,annotation,annotation_score,category,annotation_dir,annotation_score_dircor_rank_desc
str,str,str,f32,str,i8,f32
"""chr2:69505639:C:A""","""ENSG00000115977""","""cpt1_llr""",0.158203,"""missense""",1,155969.0
"""chr9:116362607:G:A""","""ENSG00000182752""","""cpt1_llr""",0.004164,"""missense""",1,324758.0
"""chr19:6415621:G:C""","""ENSG00000088247""","""cpt1_llr""",0.071917,"""missense""",1,273620.0
"""chr9:121043070:T:C""","""ENSG00000106804""","""cpt1_llr""",0.028882,"""missense""",1,317070.0
"""chr9:5069951:A:G""","""ENSG00000096968""","""cpt1_llr""",0.158219,"""missense""",1,155956.0
…,…,…,…,…,…,…
"""chr6:152373187:G:C""","""ENSG00000131018""","""cpt1_llr""",0.047801,"""missense""",1,303757.0
"""chr2:96825066:C:T""","""ENSG00000168763""","""cpt1_llr""",0.424773,"""missense""",1,36234.0
"""chr3:132462508:G:A""","""ENSG00000138246""","""cpt1_llr""",0.092406,"""missense""",1,243182.0


In [11]:
melted_anno['annotation'].value_counts(sort=True)

annotation,count
str,u64
"""esmscoremissense""",380133
"""revel_score""",362940
"""polyphen""",344747
"""esm1b_llr""",335619
"""am_pathogenicity""",333489
"""cpt1_llr""",325971


In [12]:
RAP_APPV_DIR = "project-Gyp4fvjJg0yFZ374KvP9bGFJ:/processed_data/ukbgym/avg_pheno_per_var"
LOCAL_APPV_DIR = "/home/dnanexus/data_dir"

APPV_FILE = "quant_pheno_INT_loftee_mac20_EURunrelated_miss20per_appv_small.parquet"

!dx download {RAP_APPV_DIR}/{APPV_FILE} -o {LOCAL_APPV_DIR}/{APPV_FILE}
pheno_appv = pl.scan_parquet(f"{LOCAL_APPV_DIR}/{APPV_FILE}")

# Filter to variants in annotation set and low MAC
anno_keys = anno.select(pl.col('id').unique()).lazy()

pheno_appv = (
    pheno_appv
    .join(
        anno_keys, on='id', how='semi'
    )
    .filter(
        (pl.col('n_individuals') <= mac)
    )
    .select(
        ['id', 'phenotype', 'mean_pheno_value', 'n_individuals']
    )
)

Error: path "/home/dnanexus/data_dir/quant_pheno_INT_loftee_mac20_EURunrelated
_miss20per_appv_small.parquet" already exists but -f/--overwrite was not set


In [13]:
pheno_appv.head().collect()

id,phenotype,mean_pheno_value,n_individuals
str,str,f32,i32
"""chr2:178571014:C:T""","""leg_fatfree_mass_left_int""",-0.04929,7
"""chr17:50187944:C:T""","""basal_metabolic_rate_int""",0.992802,3
"""chr2:178741274:T:C""","""seated_height_int""",0.461834,9
"""chr15:58893289:C:A""","""arm_fat_mass_right_int""",0.632569,1
"""chr2:203125973:G:C""","""seated_height_int""",0.39507,12


## Average z-score by annotation rank (sliding window)

In [ ]:
# Sliding window and bootstrap parameters
window_size = 1_000  # Number of variants in the sliding window
step_size = 1_0      # Step between sampled points (controls plot smoothness)
max_num_variants = 500_000

# LOEUF faceting
n_loeuf_bins = 4  # Number of LOEUF bins (default 4)
n_boot = 1000      # Number of bootstrap resamples across regions

# Load gnomAD constraint metrics and create LOEUF bins
gnom = (
    pl.read_csv(
        "/home/dnanexus/data_dir/gnomad.v4.1.constraint_metrics.tsv", 
        null_values=["NA"],
        separator='\t'
    )
    .filter(
        pl.col('transcript_type')=='protein_coding',
        pl.col('canonical')==True
    )
    .select(['gene_id', 'lof.oe_ci.upper'])
    .rename({'gene_id': 'region', 'lof.oe_ci.upper': 'loeuf'})
    .drop_nulls()
)

loeuf_labels = [
    f'Q{i+1} (most constrained)' if i == 0
    else f'Q{i+1} (least constrained)' if i == n_loeuf_bins - 1
    else f'Q{i+1}'
    for i in range(n_loeuf_bins)
]

loeuf_bins = gnom.with_columns(
    loeuf_bin = pl.col('loeuf').qcut(n_loeuf_bins, labels=loeuf_labels)
)

loeuf_bins

In [15]:
# --- Helper functions ---

def sliding_window_means_unweighted(zscores, bin_ends, window_size):
    """Standard prefix-sum sliding window mean (for point estimates)."""
    csum = np.empty(len(zscores) + 1, dtype=np.float64)
    csum[0] = 0.0
    np.cumsum(zscores, out=csum[1:])
    return (csum[bin_ends] - csum[bin_ends - window_size]) / window_size


def expanded_csum_at(positions, cw, cwz, z, N):
    """Compute the prefix sum of the virtual expanded array at given positions."""
    k = np.searchsorted(cw, positions, side='left')
    k_safe = np.clip(k, 1, N)
    result = cwz[k_safe - 1] + z[k_safe - 1] * (positions - cw[k_safe - 1])
    result = np.where(positions <= 0, 0.0, result)
    return result


def sliding_window_reranked(z, cw, cwz, N, total_expanded, bin_ends, window_size):
    """Sliding window mean on the virtual expanded (re-ranked) array."""
    valid_mask = bin_ends <= total_expanded
    valid_bins = bin_ends[valid_mask]
    means = np.full(len(bin_ends), np.nan, dtype=np.float64)
    if len(valid_bins) == 0:
        return means
    right = expanded_csum_at(valid_bins, cw, cwz, z, N)
    left = expanded_csum_at(valid_bins - window_size, cw, cwz, z, N)
    means[valid_mask] = (right - left) / window_size
    return means


# --- Join gene_trait_df with LOEUF bins ---
gene_trait_loeuf = gene_trait_df.join(loeuf_bins, on='region', how='inner')

bin_counts = (
    gene_trait_loeuf
    .select(['region', 'loeuf_bin'])
    .unique(subset=['region'])
    .group_by('loeuf_bin')
    .len()
)
bin_label_map = {row[0]: f"{row[0]} (n={row[1]})" for row in bin_counts.iter_rows()}

id_region = anno.select(['id', 'region']).unique().lazy()

# --- Loop over LOEUF bins ---
all_zscore_binned = []

for loeuf_bin_raw, loeuf_bin_label in sorted(bin_label_map.items()):
    print(f"\n{'='*60}")
    print(f"Processing: {loeuf_bin_label}")
    print(f"{'='*60}")

    # Filter gene_trait_df to this bin
    gt_bin = gene_trait_loeuf.filter(pl.col('loeuf_bin') == loeuf_bin_raw)
    bin_regions = gt_bin['region'].unique()

    # Filter melted_anno to this bin's regions and re-rank within the subset
    melted_anno_bin = (
        melted_anno
        .filter(pl.col('region').is_in(bin_regions))
        .with_columns(
            annotation_score_dircor = pl.col('annotation_score') * pl.col('annotation_dir').cast(pl.Float32)
        )
        .with_columns(
            annotation_score_dircor_rank_desc = pl.col('annotation_score_dircor')
                .rank(method="max", descending=True)
                .over("annotation")
                .cast(pl.Float32)
        )
        .drop('annotation_score_dircor')
    )

    # Variant z-scores for this bin
    variant_zscores = (
        pheno_appv
        .join(id_region, on='id', how='inner')
        .join(gt_bin.lazy(), on=['region', 'phenotype'], how='inner')
        .with_columns(
            mean_pheno_value = pl.col('mean_pheno_value') * pl.col('loftee_corr_dir').cast(pl.Float32)
        )
        .select(['id', 'region', 'mean_pheno_value'])
        .collect(engine='streaming')
    )
    print(f"  Variant z-scores: {variant_zscores.shape[0]} variant-gene pairs")

    # Ranked z-scores
    ranked_zscores = (
        variant_zscores.lazy()
        .join(
            melted_anno_bin.lazy().select(['id', 'region', 'annotation', 'annotation_score_dircor_rank_desc']),
            on=['id', 'region'],
            how='inner'
        )
        .sort(['annotation', 'annotation_score_dircor_rank_desc'])
        .with_columns(
            row_pos = pl.col('annotation').cum_count().over('annotation'),
        )
        .pipe(lambda df: df.filter(pl.col('row_pos') <= max_num_variants) if max_num_variants is not None else df)
        .collect(engine='streaming')
    )

    # Map regions to integer indices for fast bootstrap resampling
    bin_all_regions = sorted(gt_bin['region'].unique().to_list())
    bin_n_regions = len(bin_all_regions)
    region_idx_map = pl.DataFrame({'region': bin_all_regions, '_region_idx': np.arange(bin_n_regions, dtype=np.int32)})
    ranked_zscores = ranked_zscores.join(region_idx_map, on='region', how='left')

    annotations_list = sorted(ranked_zscores['annotation'].unique().to_list())

    # --- Prepare per-annotation sorted arrays and bin endpoints ---
    anno_arrays = {}
    for annotation in tqdm(annotations_list, desc=f'  Preparing arrays'):
        adf = ranked_zscores.filter(pl.col('annotation') == annotation)
        z = adf['mean_pheno_value'].to_numpy().astype(np.float64)
        r_idx = adf['_region_idx'].to_numpy()
        N = len(z)
        bin_ends = np.arange(window_size, N + 1, step_size, dtype=np.int64)
        anno_arrays[annotation] = {
            'zscores': z,
            'region_idx': r_idx,
            'bin_ends': bin_ends,
            'N': N,
        }

    # --- Point estimates (unweighted, on full data) ---
    point_means = {}
    for annotation in annotations_list:
        d = anno_arrays[annotation]
        point_means[annotation] = sliding_window_means_unweighted(
            d['zscores'], d['bin_ends'], window_size
        )

    # --- Re-ranking bootstrap CIs across regions ---
    rng = np.random.default_rng(42)
    boot_means = {ann: np.full((n_boot, len(anno_arrays[ann]['bin_ends'])), np.nan)
                  for ann in annotations_list}

    max_N = max(d['N'] for d in anno_arrays.values())
    cw_buf = np.empty(max_N + 1, dtype=np.float64)
    cwz_buf = np.empty(max_N + 1, dtype=np.float64)

    for b in tqdm(range(n_boot), desc=f'  Bootstrap'):
        idx = rng.choice(bin_n_regions, size=bin_n_regions, replace=True)
        region_weights = np.bincount(idx, minlength=bin_n_regions).astype(np.float64)

        for annotation in annotations_list:
            d = anno_arrays[annotation]
            N = d['N']
            z = d['zscores']
            vw = region_weights[d['region_idx']]

            cw = cw_buf[:N + 1]
            cw[0] = 0.0
            np.cumsum(vw, out=cw[1:])

            cwz = cwz_buf[:N + 1]
            cwz[0] = 0.0
            np.cumsum(z * vw, out=cwz[1:])

            total_expanded = int(cw[N])

            boot_means[annotation][b] = sliding_window_reranked(
                z, cw, cwz, N, total_expanded, d['bin_ends'], window_size
            )

    # --- Build output DataFrame with percentile-based CIs ---
    bin_results = []
    for annotation in annotations_list:
        d = anno_arrays[annotation]
        bm = boot_means[annotation]
        bin_results.append(pl.DataFrame({
            'annotation': annotation,
            'bin_end': d['bin_ends'].astype(np.float64),
            'mean_zscore': point_means[annotation].astype(np.float32),
            'ci_lower': np.nanpercentile(bm, 2.5, axis=0).astype(np.float32),
            'ci_upper': np.nanpercentile(bm, 97.5, axis=0).astype(np.float32),
        }))

    zscore_binned_bin = (
        pl.concat(bin_results)
        .sort(['annotation', 'bin_end'])
        .with_columns(loeuf_bin=pl.lit(loeuf_bin_label))
    )
    print(f"  Sliding window results: {zscore_binned_bin.shape[0]} points across {zscore_binned_bin['annotation'].n_unique()} annotations")
    all_zscore_binned.append(zscore_binned_bin)

zscore_binned = pl.concat(all_zscore_binned).sort(['loeuf_bin', 'annotation', 'bin_end'])
print(f"\nTotal: {zscore_binned.shape[0]} points across {zscore_binned['loeuf_bin'].n_unique()} LOEUF bins")
zscore_binned


Processing: Q1 (most constrained) (n=337)


/tmp/ipykernel_545252/1930616232.py:62: DeprecationWarning: `is_in` with a collection of the same datatype is ambiguous and deprecated.
Please use `implode` to return to previous behavior.

See https://github.com/pola-rs/polars/issues/22149 for more information.


  Variant z-scores: 217205 variant-gene pairs


  Bootstrap: 100%|██████████| 1000/1000 [00:27<00:00, 35.99it/s]


  Sliding window results: 116043 points across 6 annotations

Processing: Q2 (n=200)
  Variant z-scores: 88228 variant-gene pairs


...

  Bootstrap: 100%|██████████| 1000/1000 [00:03<00:00, 301.69it/s]


  Sliding window results: 13145 points across 6 annotations

Processing: Q4 (least constrained) (n=46)
  Variant z-scores: 10932 variant-gene pairs


  Bootstrap: 100%|██████████| 1000/1000 [00:01<00:00, 611.73it/s]


  Sliding window results: 5670 points across 6 annotations

Total: 183829 points across 4 LOEUF bins


annotation,bin_end,mean_zscore,ci_lower,ci_upper,loeuf_bin
str,f64,f32,f32,f32,str
"""am_pathogenicity""",1000.0,0.324696,0.250659,0.402507,"""Q1 (most constrained) (n=337)"""
"""am_pathogenicity""",1010.0,0.325055,0.25008,0.401373,"""Q1 (most constrained) (n=337)"""
"""am_pathogenicity""",1020.0,0.3144,0.245405,0.396712,"""Q1 (most constrained) (n=337)"""
"""am_pathogenicity""",1030.0,0.306898,0.240382,0.39101,"""Q1 (most constrained) (n=337)"""
"""am_pathogenicity""",1040.0,0.306688,0.237096,0.390752,"""Q1 (most constrained) (n=337)"""
…,…,…,…,…,…
"""revel_score""",10400.0,0.062936,-0.034356,0.115389,"""Q4 (least constrained) (n=46)"""
"""revel_score""",10410.0,0.061625,-0.034719,0.116178,"""Q4 (least constrained) (n=46)"""
"""revel_score""",10420.0,0.060555,-0.035274,0.116325,"""Q4 (least constrained) (n=46)"""


## Plotting

In [16]:
plt_df = (
    zscore_binned
    .drop_nans()
    .with_columns(
        log_bin_end = pl.col('bin_end').log10(),
        log_bin_end_inv = (1 / pl.col('bin_end')).log10(),
    )
    .join(
        anno_config_df.filter(pl.col('category').is_in(selected_categories)).select(['annotation', 'color', 'label', 'category']).unique(),
        on='annotation',
        how='left'
    )
    .filter(pl.col('category').is_in(selected_categories))
    .sort('bin_end')
)

plt_df

annotation,bin_end,mean_zscore,ci_lower,ci_upper,loeuf_bin,log_bin_end,log_bin_end_inv,color,label,category
str,f64,f32,f32,f32,str,f64,f64,str,str,str
"""am_pathogenicity""",1000.0,0.324696,0.250659,0.402507,"""Q1 (most constrained) (n=337)""",3.0,-3.0,"""#FFD700""","""AlphaMissense""","""missense"""
"""cpt1_llr""",1000.0,0.483308,0.383844,0.589359,"""Q1 (most constrained) (n=337)""",3.0,-3.0,"""#E63000""","""CPT-1""","""missense"""
"""esm1b_llr""",1000.0,0.24547,0.185614,0.31607,"""Q1 (most constrained) (n=337)""",3.0,-3.0,"""#FF6600""","""ESM1b""","""missense"""
"""esmscoremissense""",1000.0,0.394983,0.312025,0.482088,"""Q1 (most constrained) (n=337)""",3.0,-3.0,"""#FFA500""","""ESM1v""","""missense"""
"""polyphen""",1000.0,0.202307,0.136976,0.278238,"""Q1 (most constrained) (n=337)""",3.0,-3.0,"""peru""","""PolyPhen2""","""missense"""
…,…,…,…,…,…,…,…,…,…,…
"""esmscoremissense""",217110.0,0.019708,-0.072276,0.075979,"""Q1 (most constrained) (n=337)""",5.33668,-5.33668,"""#FFA500""","""ESM1v""","""missense"""
"""esmscoremissense""",217120.0,0.020197,-0.07514,0.076062,"""Q1 (most constrained) (n=337)""",5.3367,-5.3367,"""#FFA500""","""ESM1v""","""missense"""
"""esmscoremissense""",217130.0,0.014418,-0.073538,0.079066,"""Q1 (most constrained) (n=337)""",5.33672,-5.33672,"""#FFA500""","""ESM1v""","""missense"""


In [ ]:
x_label_ext = "Top N missense variants"
# x_label_ext = "Top N structured domain (missense) variants"
# x_label_ext = "Top N non-structured domain (missense) variants"
# x_label_ext = "Top N LIP domain (missense) variants"
# x_label_ext = "Top N intronic variants"
# x_label_ext = "Top N enhancer (pELS & dELS) variants"
# x_label_ext = "Top N promoter (PLS & CA-H3K4me3) variants"
# x_label_ext = "Top N TF binding region (TC & CA-TF) variants"

plot_df = (
    plt_df
    .filter(
        (pl.col('bin_end') <= 10_000),
        
        # (pl.col('annotation').is_in(['promoterai', 'fz_all_min']))
        # (pl.col('annotation').is_in(['promoterai', 'fz_all_max']))
        # (pl.col('annotation').is_in(['am_pathogenicity', 'esmscoremissense']))
    )
)

# Generate breaks and labels for x-axis (most extreme on the left)
max_rank = zscore_binned['bin_end'].max()
min_rank = zscore_binned['bin_end'].min()
start_exp = int(np.floor(np.log10(min_rank)))
end_exp = int(np.ceil(np.log10(max_rank)))
breaks_linear = np.arange(start_exp, end_exp + 1)
labels_sci = [f"$10^{{{int(b)}}}$" for b in breaks_linear]

color_dict = dict(zip(plt_df['label'], plt_df['color']))

(
    ggplot(
        plot_df,
        # aes(x='log_bin_end', y='mean_zscore')
        aes(x='bin_end', y='mean_zscore')
    )
    # + loftee_point
    + geom_hline(aes(yintercept=0), color='black', linetype='dotted')
    + geom_line(aes(color='label'), size=1)
    + geom_ribbon(aes(ymin='ci_lower', ymax='ci_upper', fill='label'), alpha=0.1)
    + facet_wrap("~loeuf_bin", ncol=2)
    + scale_fill_manual(values=color_dict)
    + scale_color_manual(values=color_dict)
    + labs(
        title=f"Olink z-scores stratified by LOEUF quartile\nsliding window of {window_size} variants ({n_boot} bootstrap resamples)",
        y="Mean Olink z-score",
        x=x_label_ext,
        color="Annotation",
        fill="Annotation",
    )
    # + scale_x_continuous(
    #     breaks=breaks_linear,
    #     labels=labels_sci
    # )
    + theme_minimal()
    + theme(
        figure_size=(10, 8),
        title=element_text(size=13, lineheight=1.4),
        axis_text=element_text(size=11),
        axis_title=element_text(size=13, lineheight=1.4),
        legend_text=element_text(size=11),
        legend_title=element_text(size=12),
        strip_text=element_text(size=12),
        # legend_position='bottom',
        legend_background=element_rect(fill="white", color='white', alpha=0.8),
        plot_background=element_rect(fill="white", color="white"),
    )
)

In [ ]:
# Point plot: mean z-score of top window_size variants per annotation, faceted by LOEUF bin
point_df = (
    plot_df
    .filter(pl.col('bin_end') == float(window_size))
)

# Order annotations by mean z-score (averaged across LOEUF bins), descending
label_order = (
    point_df
    .group_by('label')
    .agg(pl.col('mean_zscore').mean())
    .sort('mean_zscore', descending=False)
    ['label'].to_list()
)

color_dict = dict(zip(plt_df['label'], plt_df['color']))

(
    ggplot(
        point_df,
        aes(x='label', y='mean_zscore', color='label')
    )
    # + geom_hline(aes(yintercept=0), color='grey', linetype='dotted')
    + geom_point(size=3)
    + geom_errorbar(aes(ymin='ci_lower', ymax='ci_upper'), width=0.2)
    + facet_wrap("~loeuf_bin", ncol=2)
    + scale_color_manual(values=color_dict)
    + scale_x_discrete(limits=label_order)
    # + coord_flip()
    + labs(
        title=f"Top {window_size} variants per annotation\nstratified by LOEUF quartile ({n_boot} bootstrap resamples)",
        y="Mean Olink z-score",
        x="",
        fill="Annotation",
        color="Annotation",
    )
    + guides(color=None)
    + theme_minimal()
    + theme(
        figure_size=(8, 8),
        title=element_text(size=13, lineheight=1.4),
        axis_text=element_text(size=11),
        axis_text_x=element_text(size=0),
        axis_title=element_text(size=13),
        strip_text=element_text(size=12),
        plot_background=element_rect(fill="white", color="white"),
    )
)